In [ ]:
#%pip install torch torchvision scikit-learn seaborn scipy

# Micro-Doppler Spectrogram Classifier

## Objective
Build and train a CNN-based classifier to distinguish **four target classes** from micro-Doppler spectrograms:
- **Humans** — periodic limb-swing gait signatures
- **Vehicles** — strong bulk Doppler, minimal micro-Doppler modulation
- **Drones** — high-frequency rotor blade flashes
- **Animals** — irregular, lower-frequency motion patterns

### Pipeline
1. **Synthetic Data Generation** — simulate IQ returns → Short-Time Fourier Transform (STFT) spectrograms
2. **Dataset & DataLoader** — package spectrograms + labels for PyTorch
3. **2-D CNN Model** — convolutional feature extraction on spectrogram images
4. **Training & Validation** — Adam optimizer, cross-entropy loss, LR scheduling, early stopping
5. **Evaluation** — confusion matrix, per-class precision / recall, ROC curves


In [ ]:
# Import required libraries and set up environment
try:
    import warnings
    warnings.filterwarnings('ignore')

    # Basic data manipulation and visualization
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Signal processing
    from scipy.signal import stft

    # PyTorch imports
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader, TensorDataset

    # Machine learning utilities
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        roc_curve,
        auc,
    )

    # Reproducibility
    SEED = 42
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    print("All libraries imported successfully.")

except ImportError as e:
    print(f"Missing library – run the pip cell above first: {e}")
    raise


In [ ]:
# ------------------------------------------------------------------
# Generate synthetic micro-Doppler spectrograms for four target classes
# ------------------------------------------------------------------
# Each "sample" is a short IQ time-series that is converted to a
# spectrogram via STFT.  The micro-Doppler signature of each class
# is approximated with simple parametric models:
#   Human  : sinusoidal limb swing  (periodic, ~1-3 Hz)
#   Vehicle: strong constant Doppler shift + mild vibration
#   Drone  : high-freq blade flash  (harmonics at ~50-200 Hz)
#   Animal : random walk Doppler    (irregular, low-freq bursts)
# ------------------------------------------------------------------

CLASS_NAMES = ['Human', 'Vehicle', 'Drone', 'Animal']
NUM_CLASSES = len(CLASS_NAMES)

# Simulation parameters
FS        = 1000          # sampling rate (Hz)
DURATION  = 0.5           # seconds per sample
N_SAMPLES = 500           # IQ time-series length  (FS * DURATION)
FC        = 100           # carrier / centre frequency (Hz)
SAMPLES_PER_CLASS = 600   # total samples per class

def generate_human_iq(n=N_SAMPLES, fs=FS):
    """Sinusoidal limb-swing micro-Doppler (periodic ~1-3 Hz)."""
    t = np.arange(n) / fs
    swing_freq = np.random.uniform(1.0, 3.0)          # gait cadence
    doppler    = np.random.uniform(5, 25)              # max Doppler shift
    phase      = np.random.uniform(0, 2 * np.pi)
    fd = doppler * np.sin(2 * np.pi * swing_freq * t + phase)
    # add torso bulk Doppler
    bulk = np.random.uniform(-10, 10)
    fd += bulk
    iq = np.exp(1j * 2 * np.pi * np.cumsum(fd) / fs)
    iq += 0.15 * (np.random.randn(n) + 1j * np.random.randn(n))
    return iq

def generate_vehicle_iq(n=N_SAMPLES, fs=FS):
    """Strong bulk Doppler with engine vibration."""
    t = np.arange(n) / fs
    bulk = np.random.uniform(30, 80)                   # high constant speed
    vib_freq = np.random.uniform(20, 60)               # engine vibration
    vib_amp  = np.random.uniform(1, 4)
    fd = bulk + vib_amp * np.sin(2 * np.pi * vib_freq * t)
    iq = np.exp(1j * 2 * np.pi * np.cumsum(fd) / fs)
    iq += 0.10 * (np.random.randn(n) + 1j * np.random.randn(n))
    return iq

def generate_drone_iq(n=N_SAMPLES, fs=FS):
    """High-frequency rotor blade flashes (harmonics 50-200 Hz)."""
    t = np.arange(n) / fs
    n_rotors = np.random.choice([4, 6, 8])
    base_rpm = np.random.uniform(50, 200)
    fd = np.zeros(n)
    for h in range(1, 4):                              # first 3 harmonics
        amp = np.random.uniform(5, 15) / h
        fd += amp * np.sin(2 * np.pi * base_rpm * h * t +
                           np.random.uniform(0, 2 * np.pi))
    # add body drift
    fd += np.random.uniform(-5, 5)
    iq = np.exp(1j * 2 * np.pi * np.cumsum(fd) / fs)
    iq += 0.12 * (np.random.randn(n) + 1j * np.random.randn(n))
    return iq

def generate_animal_iq(n=N_SAMPLES, fs=FS):
    """Irregular, bursty low-frequency motion."""
    t = np.arange(n) / fs
    # random-walk Doppler
    fd = np.cumsum(np.random.randn(n) * 0.8)
    fd = np.clip(fd, -15, 15)
    # occasional burst
    burst_start = np.random.randint(0, n // 2)
    burst_len   = np.random.randint(30, 80)
    fd[burst_start:burst_start + burst_len] += np.random.uniform(5, 12)
    iq = np.exp(1j * 2 * np.pi * np.cumsum(fd) / fs)
    iq += 0.20 * (np.random.randn(n) + 1j * np.random.randn(n))
    return iq

GENERATORS = [generate_human_iq, generate_vehicle_iq,
              generate_drone_iq, generate_animal_iq]

# --- Build dataset of spectrograms ---
NPERSEG = 64   # STFT window length
NOVERLAP = 56  # overlap -> high time resolution

spectrograms = []
labels = []

for class_idx, gen_fn in enumerate(GENERATORS):
    for _ in range(SAMPLES_PER_CLASS):
        iq = gen_fn()
        _, _, Zxx = stft(iq, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP)
        mag = np.abs(Zxx)                              # magnitude spectrogram
        spectrograms.append(mag)
        labels.append(class_idx)

spectrograms = np.array(spectrograms, dtype=np.float32)
labels = np.array(labels, dtype=np.int64)

print(f"Spectrogram tensor shape : {spectrograms.shape}")
print(f"  → (samples, freq_bins, time_frames)")
print(f"Labels shape             : {labels.shape}")
print(f"Class distribution       : {dict(zip(CLASS_NAMES, np.bincount(labels)))}")


In [ ]:
# Visualise one example spectrogram per class
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(16, 4))
for cls_idx in range(NUM_CLASSES):
    idx = np.where(labels == cls_idx)[0][0]
    ax = axes[cls_idx]
    im = ax.imshow(spectrograms[idx], aspect='auto', origin='lower',
                   cmap='viridis')
    ax.set_title(CLASS_NAMES[cls_idx], fontsize=13, fontweight='bold')
    ax.set_xlabel('Time frame')
    if cls_idx == 0:
        ax.set_ylabel('Frequency bin')
    else:
        ax.set_yticks([])
fig.suptitle('Example Micro-Doppler Spectrograms by Class', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare train / validation / test splits and DataLoaders
# Spectrograms are treated as single-channel 2-D images  →  (N, 1, F, T)

X = spectrograms[:, np.newaxis, :, :]   # add channel dim
y = labels

# 70 / 15 / 15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f"Train : {X_train.shape[0]} samples")
print(f"Val   : {X_val.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")

# Convert to tensors
train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
val_ds   = TensorDataset(torch.tensor(X_val),   torch.tensor(y_val))
test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
print("\nDataLoaders created successfully.")


In [ ]:
# 2-D CNN for spectrogram classification
class MicroDopplerCNN(nn.Module):
    def __init__(self, num_classes, in_channels=1):
        super(MicroDopplerCNN, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = MicroDopplerCNN(num_classes=NUM_CLASSES).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")


In [ ]:
# Train the model
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    return running_loss / len(loader), 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return running_loss / len(loader), 100.0 * correct / total

# Hyperparameters
EPOCHS = 50
LR = 0.001
PATIENCE = 8    # early stopping patience

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  factor=0.5, patience=3,
                                                  verbose=True)

train_losses, val_losses = [], []
train_acc, val_acc = [], []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    t_loss, t_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    v_loss, v_acc = evaluate(model, val_loader, criterion, device)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    train_acc.append(t_acc)
    val_acc.append(v_acc)

    scheduler.step(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        patience_counter = 0
        best_state = model.state_dict().copy()
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  |  "
              f"Train Loss {t_loss:.4f}  Acc {t_acc:.2f}%  |  "
              f"Val Loss {v_loss:.4f}  Acc {v_acc:.2f}%")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Restore best weights
model.load_state_dict(best_state)
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")


In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Train')
plt.plot(val_acc, label='Validation')
plt.title('Model Accuracy')
plt.ylabel('Accuracy (%)')
plt.xlabel('Epoch')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Validation')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on the held-out test set
try:
    model.eval()
    all_preds = []
    all_targets = []
    all_probs = []
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds   = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs   = np.array(all_probs)

    test_accuracy = 100.0 * correct / total
    test_loss_avg = test_loss / len(test_loader)

    print(f"\nTest Accuracy : {test_accuracy:.2f}%")
    print(f"Test Loss     : {test_loss_avg:.4f}")
    print(f"\nClassification Report:\n")
    print(classification_report(all_targets, all_preds,
                                target_names=CLASS_NAMES))

except Exception as e:
    print(f"Evaluation error: {e}")
    raise


In [ ]:
# Confusion matrix visualisation
try:
    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix — Micro-Doppler Classifier', fontsize=14,
              fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    # Misclassification analysis
    incorrect_mask = (all_preds != all_targets)
    total_mis = int(np.sum(incorrect_mask))
    print("\nMisclassification Analysis:")
    print("-" * 50)
    print(f"Total misclassified samples: {total_mis} out of {len(all_targets)}")

    for i in range(NUM_CLASSES):
        mask = (all_targets == i)
        mis_mask = mask & incorrect_mask
        if np.any(mis_mask):
            unique, counts = np.unique(all_preds[mis_mask], return_counts=True)
            print(f"\n{CLASS_NAMES[i]} misclassifications:")
            for u, c in zip(unique, counts):
                print(f"  Predicted as {CLASS_NAMES[u]}: {c} samples")

except Exception as e:
    print(f"Confusion matrix error: {e}")
    raise


In [ ]:
# ROC curves (one-vs-rest) for each class
try:
    from sklearn.preprocessing import label_binarize

    y_bin = label_binarize(all_targets, classes=list(range(NUM_CLASSES)))

    plt.figure(figsize=(8, 6))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    for i in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i], lw=2,
                 label=f'{CLASS_NAMES[i]} (AUC = {roc_auc:.3f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — Micro-Doppler Classifier', fontsize=14,
              fontweight='bold')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"ROC curve error: {e}")
    raise


# Project Summary and Conclusions

## Model Performance
- Final test accuracy and loss metrics
- Per-class performance analysis (precision, recall, F1)
- Key findings from the confusion matrix — which classes are most easily confused?

## Micro-Doppler Signatures
- **Humans**: periodic limb-swing patterns are highly distinctive
- **Vehicles**: strong constant Doppler separates them clearly
- **Drones**: high-frequency rotor harmonics provide a unique fingerprint
- **Animals**: irregular, bursty motion is the hardest to separate from humans

## Recommendations
1. **Model improvements**:
   - Data augmentation (time-shift, frequency jitter, SNR variation)
   - Deeper architectures or transfer learning for real-world data
   - Multi-scale feature fusion for mixed environments

2. **Deployment considerations**:
   - Model size and real-time inference latency
   - Edge deployment on radar processing hardware
   - Continuous monitoring for domain drift

## Next Steps
- [ ] Replace synthetic data with real micro-Doppler recordings
- [ ] Add data augmentation pipeline
- [ ] Experiment with ResNet / EfficientNet backbones
- [ ] Integrate with Notebook 3 (presence heatmap) for end-to-end pipeline
